# 03B — Run the entry points step by step

Start here to inspect the model code. This notebook acts as the pipeline: it calls the same entry-point functions as the jobs, supplies local paths instead of `/opt/ml/...`, and displays the files and dataframes between steps.

Use your existing Studio kernel when it already contains pandas, scikit-learn, joblib and the selected framework. The setup cell imports this checkout directly and prints the Python and package versions it uses. No package installation, Docker, SageMaker SDK, CDK or AWS calls are needed for the included local CSV example. Start with a fresh kernel.

`execute(...)` accepts ordinary model and path arguments. Pipeline commands call these same functions; each step file's main guard remains reserved for future debugging setup. Keep this notebook open alongside the use-case scripts while you step through it.

`Prepare → Train → Evaluate → Predict`

Continue with **03C** for container-based Local runs and **03D** for managed execution. Those runs use the packages declared in the model image, which can differ from the current notebook kernel.


## Use the current kernel and this checkout

- Locate the extracted repository and load its entry-point functions directly from source.
- Keep the existing Python environment and display the installed modeling-library versions.
- Use the bundled CSV in later cells; model fitting and checkpoint files remain local.


In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import sys
import pandas as pd

# Locate the checked-out model code from the notebook's working directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'mlops-models/src/mlops_models').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository folder.')
sys.path.insert(0, str(ROOT / 'mlops-models/src'))
from mlops_models.entrypoints import prepare, train, evaluate, predict

# Show which Python interpreter, source files and libraries this run will use.
packages = []
for name in ('pandas', 'scikit-learn', 'joblib', 'autogluon.tabular', 'torch'):
    try:
        installed_version = version(name)
    except PackageNotFoundError:
        installed_version = 'not installed'
    packages.append({'package': name, 'version': installed_version})
print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)
print('Model source:', prepare.__file__)
display(pd.DataFrame(packages))


## Choose the model and local job directories

- Choose the model and read its configuration from this checkout.
- Create a new run folder containing local versions of each job's SageMaker paths.
- Select the model's example CSV already included in the repository.

Model parameters are overrides grouped by framework and task. Unspecified values come from `mlops-models/src/mlops_models/model_base/framework-defaults.json`; training saves the resolved values with its checkpoint.


In [ ]:
from datetime import datetime, timezone
import json, os, shutil
MODEL = os.environ.get('MLOPS_MODEL', 'usecase2')
MODEL_FOLDER = ROOT / 'mlops-models/src/mlops_models/models' / MODEL
config = json.loads((MODEL_FOLDER / 'model-config.json').read_text())
# Keep literal category codes such as NA when displaying saved CSVs.
CSV_OPTIONS = {'dtype': {config['identifier']: str}, 'keep_default_na': False,
               'na_values': ['', r'\N']}
SOURCE = ROOT / 'examples/data' / config['example_input']
RUN = ROOT / '.runs/notebook-entrypoints' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
PREPARED = RUN / 'prepare/opt/ml/processing/output'
TRAINING = RUN / 'train/opt/ml'
EVALUATION = RUN / 'evaluate/opt/ml/processing/output'
(TRAINING / 'input/data').mkdir(parents=True)
print('Run folder:', RUN)
display(config)

## Prepare: split rows and select features

- Execute Prepare with explicit input and output paths.
- Split the rows first, then run feature selection on the training split only.
- Inspect the selected training CSV; validation and test keep the same columns.

Usecase2 supplies its own `select_features` function. Replace that function and its ordinary imports to try different selection logic. The entry point stays the same.

In [ ]:
split_dir = prepare.execute(MODEL, SOURCE, PREPARED)
selected = pd.read_csv(split_dir / 'train/train.csv', **CSV_OPTIONS)
display(selected.head())
print('Selected columns:', list(selected.columns))

## Hand the prepared splits to training

- Copy the train and validation CSV folders into the next job's local input directory.
- Keep the test CSV outside training for the evaluation step.
- Show the input directory; managed SageMaker transfers these inputs through S3.

In [ ]:
training_input = TRAINING / 'input/data'
for name in ('train', 'validation'):
    shutil.copytree(split_dir / name, training_input / name)
print('Training inputs:', training_input)
print('The test split stays with Prepare for evaluation.')

## Train: run the use-case trainer and save outputs

- Execute the Train entry point with local input, model and auxiliary-output directories.
- Run the selected use-case function, including its trainer and preprocessing overrides.
- Display resolved parameters, metadata, exact splits and the prepared X/y passed to fitting.

Inside training, `fit_result` is a `FitResult(predictor, selected_model)`: the trained Python object and its name. The framework saves that predictor to disk; this entry point returns the checkpoint path. The object can then leave memory because prediction reloads the saved checkpoint. The metadata records the selected model name and split row counts.

Prepare has already split the rows. The trainer fits preprocessing on the training split, and its `prepare()` method returns `PreparedTrainingData`: pandas `X_train`, `y_train`, `X_validation` and `y_validation`. Every framework receives that object and performs its own conversion. The test split stays outside fitting.

In [ ]:
checkpoint_path = train.execute(MODEL, training_input, TRAINING / 'model', TRAINING / 'output/data')
metadata = json.loads((checkpoint_path / 'model-metadata.json').read_text())
splits = {name: pd.read_csv(split_dir / name / f'{name}.csv', **CSV_OPTIONS)
          for name in ('train', 'validation', 'test')}
trained_config = json.loads((checkpoint_path / 'model-config.json').read_text())
display(trained_config['parameters'])
display(metadata)
display(pd.DataFrame([{'split': name, 'rows': len(frame)} for name, frame in splits.items()]))
display(splits['train'].head())
X_train = pd.read_csv(TRAINING / 'output/data/modeling/train/X.csv', index_col='row_index')
y_train = pd.read_csv(TRAINING / 'output/data/modeling/train/y.csv', index_col='row_index')
print('Exact prepared training inputs:', X_train.shape, y_train.shape)
display(X_train.head())

## Evaluate: reload the saved model and test split

- Execute Evaluate against the actual checkpoint and retained test CSV.
- Reload the fitted preprocessing and score without fitting another model.
- Display the evaluation report produced by the entry point.

In [ ]:
report = evaluate.execute(MODEL, checkpoint_path, EVALUATION,
                          input_path=split_dir / 'test/test.csv')
display(report)

## Predict: inspect the checkpoint’s predictions directly

- Run the direct prediction entry point against the input CSV.
- Use the saved model’s preprocessing and use-case output formatting.
- Inspect the predictions before adding the Batch Transform server in 03C.

In [ ]:
prediction_path = predict.execute(MODEL, SOURCE, checkpoint_path, RUN / 'predict/predictions.csv')
predictions = pd.read_csv(prediction_path, **CSV_OPTIONS)
display(predictions.head())
print('Prediction rows:', len(predictions))

To experiment, edit the relevant use-case function and restart the kernel before rerunning so the new imports are used. The first cell chooses a new run directory.

These folders simulate job input/output mounts. No S3 upload, registry call or hosted model is performed here. Managed training uploads `/opt/ml/model` as `model.tar.gz` and `/opt/ml/output/data` as an auxiliary archive. Evaluate reads the model archive and Prepare's separate test CSV. **03C** exercises those container steps and the actual batch server.

Feature selection and fitted preprocessing use training rows only. This small synthetic dataset demonstrates the workflow; it does not establish production model quality.